# MMDP-OD-RTDP Main Report

This notebook serves as the empirical report for comparing **Baseline Real-Time Dynamic Programming (RTDP)** against **Operator Decomposition (OD) RTDP** in stochastic Multi-Agent Pathfinding environments (MMDP).

By dynamically running the core framework modules, this notebook directly produces the empirical metrics required to compare the two planning approaches under constrained computational budgets.

### 1. Repository Setup & Module Import
The project is structured as a modular Python framework. The code below checks if it is running in Google Colab and fetches the repository if necessary.

In [ ]:
import sys
import os
from pathlib import Path
import time
from IPython.display import display, Markdown

REPO_URL = "https://github.com/t-rays/MMDP-OD-RTDP-PROJECT.git"
REPO_NAME = "MMDP-OD-RTDP-PROJECT"

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    sys.path.append('src')
else:
    src_path = str(Path('.').resolve() / 'src')
    if src_path not in sys.path:
        sys.path.append(src_path)
        
from grid_mmdp import GridMMDP, MMDPConfig
from map_creator import create_map_instance
from heuristic import ShortestPathHeuristic
from baseline_rtdp import BaselineRTDP, RTDPConfig
from od_rtdp import OperatorDecompositionRTDP
from evaluation import evaluate_policy, EvaluationConfig

print("✅ Framework loaded successfully.")

### 2. Experiment Definition
We define a reusable function that runs both algorithms on a given map. The function enforces a strict time limit to demonstrate how OD RTDP's reduced branching factor allows for deeper exploration and more Bellman backups within the same time boundary.

In [ ]:
def run_comparison(map_path: str, n_agents: int, time_limit: float = 5.0, episodes: int = 50):
    # 1. Initialize the Environment
    map_instance = create_map_instance(map_path, scenario_number=1, task_offset=0, n_agents=n_agents)
    mdp_config = MMDPConfig(slip_to_stay_probability=0.20)
    mdp = GridMMDP(map_instance, mdp_config)
    heuristic = ShortestPathHeuristic(mdp)
    
    print(f"\n{'='*50}\nRunning Map: {map_instance.map_name} ({n_agents} Agents)\n{'='*50}")
    
    # 2. Configure Planners
    config = RTDPConfig(time_limit_seconds=time_limit, seed=42)
    eval_config = EvaluationConfig(episodes=episodes, seed=101)
    
    # 3. Baseline RTDP
    baseline_planner = BaselineRTDP(mdp, heuristic, config)
    print("Running Baseline RTDP...")
    baseline_result = baseline_planner.solve()
    baseline_eval = evaluate_policy(mdp, baseline_planner, eval_config)
    
    # 4. OD RTDP
    od_planner = OperatorDecompositionRTDP(mdp, heuristic, config)
    print("Running OD RTDP...")
    od_result = od_planner.solve()
    od_eval = evaluate_policy(mdp, od_planner, eval_config)
    
    # Return gathered metrics for the table
    return {
        "map": map_instance.map_name,
        "agents": n_agents,
        "baseline": {
            "trials": baseline_result.trials_completed,
            "backups": baseline_result.bellman_backups,
            "success": baseline_eval.summary.success_rate,
            "steps": baseline_eval.summary.mean_steps_successful_episodes
        },
        "od": {
            "trials": od_result.trials_completed,
            "backups": od_result.bellman_backups,
            "success": od_eval.summary.success_rate,
            "steps": od_eval.summary.mean_steps_successful_episodes
        }
    }

### 3. Execution
We run the comparison across a few different scenarios (e.g., an empty grid and a diagnostic crossing map) giving each planner exactly 5.0 seconds.

In [ ]:
results = []

# Run 1: Simple Empty 8x8 Map (3 Agents)
results.append(run_comparison('maps/empty-8-8', n_agents=3, time_limit=5.0))

# Run 2: Diagnostic Crossing 9x9 Map (3 Agents)
results.append(run_comparison('maps/diagnostic/crossing-9-9', n_agents=3, time_limit=5.0))

print("\n✅ Experiments completed.")

### 4. Empirical Comparison Results
The table below is generated dynamically from the execution metrics. It highlights the vast difference in the number of Bellman Backups completed, which directly correlates to the quality (success rate) of the resulting policy.

In [ ]:
def generate_markdown_table(results):
    table = "| Map | Agents | Algorithm | Trials | Bellman Backups | Success Rate | Mean Steps |\n"
    table += "|---|---|---|---|---|---|---|\n"
    
    for res in results:
        # Baseline Row
        b = res['baseline']
        table += f"| **{res['map']}** | {res['agents']} | **Baseline RTDP** | {b['trials']:,} | {b['backups']:,} | {b['success']*100:.1f}% | {b['steps']:.2f} |\n"
        
        # OD Row
        o = res['od']
        table += f"| | | **OD RTDP** | {o['trials']:,} | {o['backups']:,} | {o['success']*100:.1f}% | {o['steps']:.2f} |\n"
    
    return table

# Render the table
display(Markdown(generate_markdown_table(results)))